In [13]:
import numpy as np

# A 2x2 Markov operator (vector AR(1)):   x_{t+1} = L x_t + eps_t
#   x0 is persistent (0.8) and is also fed by x1 through the off-diagonal;
#   x1 is nearly white (0.1).
L = np.array([[0.8, 0.2],
              [0.0, 0.1]])

# What must the forcing look like to keep BOTH variances at 1?
# Stationarity:  C = L C L^T + Q.   Demand C = I  =>  Q = I - L L^T
Q = np.eye(2) - L @ L.T

# The forcing matrix is any A with A A^T = Q, so that eps_t = A w_t, w_t ~ N(0, I).
# The Cholesky factor is the usual choice.
A = np.linalg.cholesky(Q)

np.set_printoptions(precision=4, suppress=True)
print("L =\n", L)
print("\nL L^T =\n", L @ L.T)
print("\nQ = I - L L^T =\n", Q)
print("\nA = chol(Q) =\n", A)
print("\nA A^T == Q ?", np.allclose(A @ A.T, Q))
print("singular values of L:", np.linalg.svd(L, compute_uv=False),
      "-> all <= 1, so Q is positive semi-definite and A exists")

L =
 [[0.8 0.2]
 [0.  0.1]]

L L^T =
 [[0.68 0.02]
 [0.02 0.01]]

Q = I - L L^T =
 [[ 0.32 -0.02]
 [-0.02  0.99]]

A = chol(Q) =
 [[ 0.5657  0.    ]
 [-0.0354  0.9944]]

A A^T == Q ? True
singular values of L: [0.825 0.097] -> all <= 1, so Q is positive semi-definite and A exists


## Multivariate Markov chain with unit-variance components

The cell above sets up a **vector AR(1)** (a two-variable Markov chain),

$$\mathbf{x}_{t+1} = L\,\mathbf{x}_t + \boldsymbol{\varepsilon}_t,
\qquad \boldsymbol{\varepsilon}_t \sim \mathcal{N}(\mathbf{0},\,Q),
\qquad
L=\begin{pmatrix}0.8 & 0.2\\[2pt] 0 & 0.1\end{pmatrix},$$

and asks the design question: **what does the stochastic forcing have to look like so that both components keep unit variance?**

### Solving for the forcing

Because $\boldsymbol{\varepsilon}_t$ is independent of $\mathbf{x}_t$, the stationary covariance $C=\langle\mathbf{x}\mathbf{x}^{T}\rangle$ obeys the **discrete Lyapunov equation**

$$C = L\,C\,L^{T} + Q .$$

Setting $C=I$ (unit variance in both components, and uncorrelated) and solving for $Q$:

$$\boxed{\,Q = I - L L^{T}\,}$$

so the forcing must exactly replace the variance that $L$ removes at each step. For the $L$ above,

$$L L^{T}=\begin{pmatrix}0.68 & 0.02\\ 0.02 & 0.01\end{pmatrix}
\quad\Longrightarrow\quad
Q=\begin{pmatrix}0.32 & -0.02\\ -0.02 & 0.99\end{pmatrix}.$$

Two things are worth noticing. The damping is very unequal — the persistent component only needs $0.32$ of noise variance while the nearly-white one needs $0.99$ — and $Q$ is **not diagonal**: the off-diagonal $-0.02$ is there to cancel the correlation that the coupling term $0.2\,x_1$ would otherwise inject into $x_0$.

The forcing is drawn as $\boldsymbol{\varepsilon}_t = A\mathbf{w}_t$ with $\mathbf{w}_t\sim\mathcal{N}(\mathbf{0},I)$ and any $A$ such that $AA^{T}=Q$; the Cholesky factor is the usual choice,

$$A=\begin{pmatrix}0.5657 & 0\\ -0.0354 & 0.9944\end{pmatrix}.$$

### Relation to the scalar case

**This is the matrix version of the scalar red-noise normalisation.** For one variable $L\to a$, $Q\to 1-a^{2}$, recovering the familiar

$$x_{t+1} = a\,x_t + \sqrt{1-a^{2}}\;\varepsilon_t ,$$

which is the unit-variance AR(1) used throughout the Regression chapter.

> **Caveat.** $Q=I-LL^{T}$ is a valid covariance only if it is positive semi-definite, which requires the largest **singular value** of $L$ to be $\le 1$ — a stricter condition than stability ($|\lambda_i|<1$). Here the singular values are $0.825$ and $0.097$, so we are safe, even though $L$ is non-normal ($LL^{T}\neq L^{T}L$). The two conditions come apart for strongly non-normal $L$, as the last cell shows.

The next cell integrates the chain and checks that the construction really does deliver unit variances — and, since $C=I$ makes the lag-1 cross-covariance $\langle\mathbf{x}_{t+1}\mathbf{x}_t^{T}\rangle = LC = L$, that each component's lag-1 autocorrelation equals the corresponding diagonal entry of $L$.

In [14]:
from scipy.linalg import solve_discrete_lyapunov

rng = np.random.default_rng(0)

# L, Q and A come from the first cell.
assert np.all(np.linalg.eigvalsh(Q) >= 0), \
    "max singular value of L exceeds 1: no unit-variance forcing exists"

# --- integrate the chain
n, burn = 500_000, 1_000
x = np.zeros((2, n))
w = rng.standard_normal((2, n))
for t in range(1, n):
    x[:, t] = L @ x[:, t - 1] + A @ w[:, t]

xs = x[:, burn:]

print("eigenvalues of L   :", np.linalg.eigvals(L).real)
print("singular values    :", np.linalg.svd(L, compute_uv=False))
print("L normal?          :", np.allclose(L @ L.T, L.T @ L))
print("\nanalytic stationary covariance C (Lyapunov) =\n", solve_discrete_lyapunov(L, Q))
print("\nempirical covariance (n = %d) =\n" % (n - burn), np.cov(xs))

# Because C = I, the lag-1 cross-covariance is <x_{t+1} x_t^T> = L C = L,
# so each component's lag-1 autocorrelation should equal the diagonal of L.
for i in range(2):
    s = xs[i]
    print(f"\nx{i}:  var = {s.var():.4f}  (target 1)"
          f"   lag-1 autocorr = {np.corrcoef(s[:-1], s[1:])[0, 1]:.4f}  (target {L[i, i]})")

eigenvalues of L   : [0.8 0.1]
singular values    : [0.825 0.097]
L normal?          : False

analytic stationary covariance C (Lyapunov) =
 [[1. 0.]
 [0. 1.]]

empirical covariance (n = 499000) =
 [[ 0.9992 -0.0022]
 [-0.0022  1.0005]]

x0:  var = 0.9992  (target 1)   lag-1 autocorr = 0.7992  (target 0.8)

x1:  var = 1.0005  (target 1)   lag-1 autocorr = 0.1001  (target 0.1)


In [12]:
# Counterexample: stability is NOT enough. This L is stable (both eigenvalues 0.5)
# but strongly non-normal, so I - L L^T is indefinite and no forcing can give
# unit variance in both components -- transient growth from the off-diagonal
# coupling amplifies more than the noise can be reduced to compensate.

L_bad = np.array([[0.5, 3.0],
                  [0.0, 0.5]])
Q_bad = np.eye(2) - L_bad @ L_bad.T

print("eigenvalues        :", np.linalg.eigvals(L_bad).real, " -> stable")
print("max singular value :", np.linalg.svd(L_bad, compute_uv=False)[0], " -> > 1")
print("eig(I - L Lt)      :", np.linalg.eigvalsh(Q_bad), " -> not PSD")

try:
    np.linalg.cholesky(Q_bad)
except np.linalg.LinAlgError as e:
    print("\ncholesky failed, as expected:", e)

eigenvalues        : [0.5 0.5]  -> stable
max singular value : 3.08113883008419  -> > 1
eig(I - L Lt)      : [-8.49341649  0.99341649]  -> not PSD

cholesky failed, as expected: Matrix is not positive definite
